# TimeDepBurgers2D
Notebook to implement and test the time-dependent 2D Burgers' equation.  
Author: Alejandro Diaz  
Date: 1/2/2024

In [ ]:
import sys
with open("./../../../PATHS.txt") as file:
  paths = file.read().splitlines()
sys.path.extend(paths)

In [ ]:
from dd_nm_rom import env
env.set(
  backend="numpy",
  device="cpu",
  device_idx=0,
  nb_threads=8,
  epsilon=1e-10,
  floatx="float64",
  seed=0
)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

from IPython.display import Image
from IPython.display import HTML

In [ ]:
from dd_nm_rom import postproc
from dd_nm_rom import fom as fom_mod
from dd_nm_rom import field as field_mod
from dd_nm_rom.elements import mesh as mesh_mod

In [ ]:
# DD Mesh
nx_intr = 48
ny_intr = 48
lx_sub = 0.5
ly_sub = 0.5
x0 = 0.0
y0 = 0.0
n_sub_x = 2
n_sub_y = 2
# Time grid
dt = 0.03
nt = 2
t_lim = [0, dt*nt]
# PDE
viscosity = 1e-3

In [ ]:
mesh = mesh_mod.MeshDD(
  nx_intr=nx_intr,
  ny_intr=ny_intr,
  lx_sub=lx_sub,
  ly_sub=ly_sub,
  x0=x0,
  y0=y0,
  n_sub_x=n_sub_x,
  n_sub_y=n_sub_y,
  with_bounds=True
)
mesh.build()
X, Y = mesh.grid

In [ ]:
path_to_figs = "./../../../../../run/figures/"
fig_dir = path_to_figs + f'/unsteady/nx_intr_{nx_intr}_ny_intr_{ny_intr}_lx_{lx_sub}_ly_{ly_sub}/fom/'
os.makedirs(fig_dir, exist_ok=True)

# Single sin peak

In [ ]:
fig_dir_i = fig_dir + "/sin_peak_2by2/"
os.makedirs(fig_dir_i, exist_ok=True)

In [ ]:
field = field_mod.SinPeak(mesh=mesh, mu_lim=[0.9,1.1], bc_type="periodic")
field.set_params(mu=field.sample_design_space())
U0 = field.u()
V0 = field.v()

#### 2D Burgers equation

In [ ]:
fom = fom_mod.Burgers2D(
  nu=viscosity,
  mesh=mesh
)
fom.build(field)

In [ ]:
postproc.plot_field(
  x=X,
  y=Y,
  z=U0,
  label='$u$',
  show_labels=False,
  cmap="viridis",
  filename=fig_dir_i + '/u0.png',
  save=False,
  show=True
)
postproc.plot_field(
  x=X,
  y=Y,
  z=V0,
  label='$v$',
  cmap="viridis",
  filename=fig_dir_i + '/v0.png',
  save=False,
  show=True
)

In [ ]:
# Steady
x0 = np.concatenate([U0.reshape(-1), V0.reshape(-1)])
uv, rhs, converged = fom.solve(x0, dt=dt, nt=nt, steady=False, tol=1e-8, maxit=20, stepsize_min=1e-10, verbose=True)
print("RUNTIME:", fom.runtime)

In [ ]:
UU = uv["u"].T.reshape(-1, mesh.n["y"], mesh.n["x"])
anim = postproc.animate(
  x=X,
  y=Y,
  z=UU,
  lim=[UU.min(), UU.max()],
  label='$u$',
  show_labels=True,
  frames=len(UU),
  fps=10,
  filename=fig_dir_i + '/u_state.gif',
  dpi=600,
  save=False
)
HTML(anim.to_jshtml())

In [ ]:
VV = uv["v"].T.reshape(-1, mesh.n["y"], mesh.n["x"])
anim = postproc.animate(
  x=X,
  y=Y,
  z=VV,
  lim=[VV.min(), VV.max()],
  label='$v$',
  frames=len(VV),
  fps=10,
  filename=fig_dir_i + '/v_state.gif',
  dpi=600,
  save=False
)
HTML(anim.to_jshtml())

#### 2D Burgers equation with domain-decomposition

In [ ]:
ddmdl = fom_mod.DDBurgers2D(fom, constraint_type='strong')
ddmdl.build()

In [ ]:
# Steady
uv_dd, lambdas, rhs, converged = ddmdl.solve(
  x0=ddmdl.get_init_sol(x=x0),
  dt=dt,
  nt=nt,
  steady=False,
  tol=1e-8,
  maxit=20,
  stepsize_min=1e-10,
  verbose=True
)
print("RUNTIME:", ddmdl.runtime)

In [ ]:
UU_dd = uv_dd["res"]["u"].T.reshape(-1, mesh.n["y"], mesh.n["x"])
anim = postproc.animate(
  x=X,
  y=Y,
  z=UU_dd,
  lim=[UU.min(), UU.max()],
  label='$u$',
  frames=len(UU),
  fps=10,
  filename=fig_dir_i + '/u_dd_state.gif',
  dpi=600,
  save=False
)
HTML(anim.to_jshtml())

In [ ]:
VV_dd = uv_dd["res"]["v"].T.reshape(-1, mesh.n["y"], mesh.n["x"])
anim = postproc.animate(
  x=X,
  y=Y,
  z=VV_dd,
  lim=[VV.min(), VV.max()],
  label='$v$',
  frames=len(VV),
  fps=10,
  filename=fig_dir_i + '/v_dd_state.gif',
  dpi=600,
  save=False
)
HTML(anim.to_jshtml())

In [ ]:
UU_dd_err = np.abs(uv_dd["res"]["u"] - uv["u"])
UU_dd_err = UU_dd_err.T.reshape(-1, mesh.n["y"], mesh.n["x"])
anim = postproc.animate(
  x=X,
  y=Y,
  z=UU_dd_err,
  lim=[UU_dd_err.min(), UU_dd_err.max()],
  label="$|\hat{u}-u|$",
  frames=len(UU_dd_err),
  fps=10,
  filename=fig_dir_i + '/u_dd_state_err.gif',
  dpi=600,
  save=False
)
HTML(anim.to_jshtml())

In [ ]:
VV_dd_err = np.abs(uv_dd["res"]["v"] - uv["v"])
VV_dd_err = VV_dd_err.T.reshape(-1, mesh.n["y"], mesh.n["x"])
anim = postproc.animate(
  x=X,
  y=Y,
  z=VV_dd_err,
  lim=[VV_dd_err.min(), VV_dd_err.max()],
  label="$|\hat{v}-v|$",
  frames=len(VV_dd_err),
  fps=10,
  filename=fig_dir_i + '/v_dd_state_err.gif',
  dpi=600,
  save=False
)
HTML(anim.to_jshtml())

# Multi sin peak

In [ ]:
fig_dir_i = fig_dir + "/sin_multi_peak_2by2/"
os.makedirs(fig_dir_i, exist_ok=True)

In [ ]:
field = field_mod.SinMultiPeak(mesh=mesh, mu_lim=[0.5,1.5], forced_config=[1,0,0,0])

In [ ]:
field.set_params(mu=field.sample_design_space())
U0 = field.u()
V0 = field.v()

In [ ]:
postproc.plot_field(
  x=X,
  y=Y,
  z=U0,
  label='$u$',
  cmap="viridis",
  filename=fig_dir_i + '/u0.png',
  save=False,
  show=True
)
postproc.plot_field(
  x=X,
  y=Y,
  z=V0,
  label='$v$',
  cmap="viridis",
  filename=fig_dir_i + '/v0.png',
  save=False,
  show=True
)

In [ ]:
# Steady
x0 = np.concatenate([U0.reshape(-1), V0.reshape(-1)])
uv, rhs, converged = fom.solve(x0, dt=dt, nt=nt, steady=False, tol=1e-8, maxit=20, stepsize_min=1e-10, verbose=True)
print("RUNTIME:", fom.runtime)

In [ ]:
UU = uv["u"].T.reshape(-1, mesh.n["y"], mesh.n["x"])
anim = postproc.animate(
  x=X,
  y=Y,
  z=UU,
  lim=[UU.min(), UU.max()],
  label='$u$',
  frames=len(UU),
  fps=10,
  filename=fig_dir_i + '/u_state.gif',
  dpi=600,
  save=False
)
HTML(anim.to_jshtml())

In [ ]:
VV = uv["v"].T.reshape(-1, mesh.n["y"], mesh.n["x"])
anim = postproc.animate(
  x=X,
  y=Y,
  z=VV,
  lim=[VV.min(), VV.max()],
  label='$v$',
  frames=len(VV),
  fps=10,
  filename=fig_dir_i + '/v_state.gif',
  dpi=600,
  save=False
)
HTML(anim.to_jshtml())

In [ ]:
fig_dir_i = fig_dir + "/sin_multi_peak_3by3/"
os.makedirs(fig_dir_i, exist_ok=True)

In [ ]:
mesh_cfg = mesh.get_config()
mesh_cfg["n_sub_x"] = 3
mesh_cfg["n_sub_y"] = 3
mesh_big = mesh_mod.MeshDD(**mesh_cfg)
mesh_big.build()
Xb, Yb = mesh_big.grid

In [ ]:
forced_config = np.array([1,0,0,0,1,0,0,0,1])
field = field_mod.SinMultiPeak(mesh_big, mu_lim=[0.5,1.5], forced_config=forced_config)
mu = field.sample_design_space()
mu[forced_config.astype(bool)] = 1.5

In [ ]:
field.set_params(mu=mu)
U0 = field.u()
V0 = field.v()

In [ ]:
postproc.plot_field(
  x=Xb,
  y=Yb,
  z=U0,
  label='$u$',
  cmap="viridis",
  filename=fig_dir_i + '/u0.png',
  save=False,
  show=True
)
postproc.plot_field(
  x=Xb,
  y=Yb,
  z=V0,
  label='$v$',
  cmap="viridis",
  filename=fig_dir_i + '/v0.png',
  save=False,
  show=True
)

In [ ]:
fom = fom_mod.Burgers2D(
  nu=viscosity,
  mesh=mesh_big
)
fom.build(field)

In [ ]:
# Steady
x0 = np.concatenate([U0.reshape(-1), V0.reshape(-1)])
uv, rhs, converged = fom.solve(x0, dt=dt, nt=nt, steady=False, tol=1e-8, maxit=20, stepsize_min=1e-10, verbose=True)
print("RUNTIME:", fom.runtime)

In [ ]:
UU = uv["u"].T.reshape(-1, mesh_big.n["y"], mesh_big.n["x"])
anim = postproc.animate(
  x=Xb,
  y=Yb,
  z=UU,
  lim=[UU.min(), UU.max()],
  label='$u$',
  frames=len(UU),
  fps=10,
  filename=fig_dir_i + '/u_state.gif',
  dpi=600,
  save=False
)
HTML(anim.to_jshtml())

In [ ]:
VV = uv["v"].T.reshape(-1, mesh_big.n["y"], mesh_big.n["x"])
anim = postproc.animate(
  x=Xb,
  y=Yb,
  z=VV,
  lim=[VV.min(), VV.max()],
  label='$v$',
  frames=len(VV),
  fps=10,
  filename=fig_dir_i + '/v_state.gif',
  dpi=600,
  save=False
)
HTML(anim.to_jshtml())